In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

df = pd.read_csv("C:\\Users\\91906\\OneDrive\\Desktop\\churn-prediction\\data\\telco_churn.csv.csv")

# Fix TotalCharges again
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['TotalCharges'].median()
)

print(df.shape)

df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Tenure groups
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['New', 'Developing', 'Established', 'Loyal']
)

# Average monthly spending
df['avg_monthly_spend'] = (
    df['TotalCharges'] / (df['tenure'] + 1)
)

# Number of services used
service_cols = [
    'PhoneService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df['num_services'] = df[service_cols].apply(
    lambda x: (x == 'Yes').sum(),
    axis=1
)

# High value customer flag
df['is_high_value'] = (
    df['MonthlyCharges'] > 70
).astype(int)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,avg_monthly_spend,num_services,is_high_value
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,New,14.925000,1,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,Established,53.985714,3,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,New,36.050000,3,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,Established,40.016304,3,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,New,50.550000,1,1


In [3]:
# Remove customer ID
df.drop('customerID', axis=1, inplace=True)

# Encode binary columns
le = LabelEncoder()

binary_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling',
    'Churn'
]

for col in binary_cols:
    df[col] = le.fit_transform(df[col])

print(df.head())

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

      MultipleLines InternetService OnlineSecurity OnlineBackup  ...  \
0  No phone service             DSL             No          Yes  ...   
1                No             DSL            Yes           No  ...   
2                No             DSL            Yes          Yes  ...   
3  No phone service             DSL            Yes           No  ...   
4                No     Fiber optic             No           No  ...   

         Contract PaperlessBilling              PaymentMethod MonthlyCharges  \
0  Month-to-month                1           Electronic chec

In [4]:
# Convert categorical columns into dummy variables
df = pd.get_dummies(df, drop_first=True)

print("Final Shape:", df.shape)

df.head()

Final Shape: (7043, 37)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_Developing,tenure_group_Established,tenure_group_Loyal
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,True,False,False,False,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,True,False,False,False,True,False,True,False
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,True,False,False,False
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,False,False,True,False,False,False,False,False,True,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,True,False,False,False,False


In [5]:
from sklearn.model_selection import train_test_split

# Features and target
X = df.drop('Churn', axis=1)

y = df['Churn']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Before SMOTE:")
print(y_train.value_counts())

# Apply SMOTE
sm = SMOTE(random_state=42)

X_train_res, y_train_res = sm.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_res).value_counts())

Before SMOTE:
Churn
0    4139
1    1495
Name: count, dtype: int64

After SMOTE:
Churn
0    4139
1    4139
Name: count, dtype: int64


In [6]:
# Scale features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_res)

X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)

print(X_test_scaled.shape)

(8278, 36)
(1409, 36)


In [7]:
# Save scaler
pickle.dump(
    scaler,
    open('../models/scaler.pkl', 'wb')
)

# Save feature names
pickle.dump(
    list(X.columns),
    open('../models/feature_names.pkl', 'wb')
)

print("Scaler and feature names saved!")

Scaler and feature names saved!


In [8]:
# Save processed datasets
np.save('../data/X_train.npy', X_train_scaled)

np.save('../data/X_test.npy', X_test_scaled)

np.save('../data/y_train.npy', y_train_res)

np.save('../data/y_test.npy', y_test)

print("All processed datasets saved!")

All processed datasets saved!
